In [ ]:
import pandas as pd                               # Importamos las librerias necesarias
from sqlalchemy import create_engine
from dotenv import load_dotenv 
import os

load_dotenv("../rock/.env")                       #Cargamos nuestras contraseñas en el .env
SQL_pass = os.getenv("SQL")

ModuleNotFoundError: No module named 'sqlalchemy'

In [ ]:
def conectar_db():                           # Funcion para conectarnos a MySQl
    usuario = 'root'
    password = SQL_pass
    host = "127.0.0.1"
    db = 'music_db' 
    
    engine = create_engine(f'mysql+mysqlconnector://{usuario}:{password}@{host}/{db}')     # Usamos alchemy
    return engine

In [ ]:
def subir_artistas(path_csv1, engine):                 # Funcion para subir la tabla artists
    df = pd.read_csv(path_csv1)                        # Convertimos a pandas
    
    df = df.rename(columns={                          # Ponemos los mismos nombres para que no haya errores
        'id_artist': 'id_artist',
        'artist': 'artist_name',
        'bio': 'artist_bio',
        'listeners': 'artist_listeners',
        'playcount': 'artist_playcount',
        'similar_artist': 'similar_artist'
    })
    df['artist_name'] = df['artist_name'].str.strip()           # Quitamos espacios para que no tengamos errores al subir los datos
    
    df = df.drop_duplicates(subset=['id_artist'])               # Eliminamos duplicados
    
    try:                                                     # Intentamos subir los datos si no estan en la base de datos
        artistas_en_db = pd.read_sql('SELECT id_artist FROM artists', con=engine)
        df = df[~df['id_artist'].isin(artistas_en_db['id_artist'])]
    except:                                                 # Si no puede subirlo, continua
         pass

    if not df.empty:
        df.to_sql('artists', con=engine, if_exists='append', index=False)
        print(f"✅ Se han subido {len(df)} artistas nuevos.")
    else:
        print("ℹ️ No hay artistas nuevos para subir.")
    
    # Devolvemos todos los artistas que hay en la DB
    return pd.read_sql('SELECT * FROM artists', con=engine)

In [ ]:

def procesar_y_subir_tracks(lista_csv_generos, engine):
  
    print("Obteniendo IDs de artistas desde SQL...")                              # Traemos los artistas directamente de la base de datos para estar seguros
    df_artists_db = pd.read_sql('SELECT id_artist, artist_name FROM artists', con=engine)

    mapeo_generos = {                                             # Diccionario de géneros
        "rock": 1,
        "indie": 2,
        "latin": 3,
        "reggaeton": 4,
        "hip-hop": 5
    }

    for archivo in lista_csv_generos:                        # Iteramos para que acceda a cada uno de los csv que le damos en la lista
        try:
            print(f"Procesando {archivo}...")
            df_tracks = pd.read_csv(archivo)
            
            df_tracks['artist'] = df_tracks['artist'].str.strip()                         # Limpieza y preparación para que no haya errores de separaciones o mayusculas
            df_tracks['genre'] = df_tracks['genre'].str.strip().str.lower()

            df_unido = df_tracks.merge(df_artists_db,                                          # Cruzar con los artistas de la DB
                                       left_on='artist', 
                                       right_on='artist_name', 
                                       how='left')

            df_unido['id_genre'] = df_unido['genre'].map(mapeo_generos)                        # Mapear el ID del género

            df_final = df_unido[['id_artist', 'id_genre', 'track_name', 'year']]           # Seleccionar columnas exactas del esquema SQL
            
            df_final = df_final.dropna(subset=['id_artist', 'id_genre'])                   # Quitar nulos

            try:                                                                       # Si hay duplicados
                existentes = pd.read_sql('SELECT id_artist, track_name FROM tracks', con=engine)
        
                df_final['check'] = df_final['id_artist'].astype(str) + df_final['track_name']       # Combinamos ID y Nombre para identificar la canción única
                existentes['check'] = existentes['id_artist'].astype(str) + existentes['track_name']
                
                df_final = df_final[~df_final['check'].isin(existentes['check'])]
                df_final = df_final.drop(columns=['check'])
            except:
                pass                                                                          # Si la tabla está vacía, no hay nada que filtrar

            # Subir a SQL
            if not df_final.empty:
                df_final.to_sql('tracks', con=engine, if_exists='append', index=False)
                print(f"✅ {len(df_final)} canciones nuevas subidas desde {archivo}.")
            else:
                print(f"ℹ️ No había canciones nuevas en {archivo}.")

        except Exception as e:
            print(f"❌ Error procesando {archivo}: {e}")

In [ ]:
engine = conectar_db()                      # Conectar

In [ ]:
df_maestro_artistas = subir_artistas('resultado_reggaeton.csv', engine)             # Subir Artistas

✅ Se han subido 62 artistas nuevos.


In [ ]:
df_maestro_artistas = subir_artistas('resultado_latin.csv', engine)

✅ Se han subido 296 artistas nuevos.


In [ ]:
df_maestro_artistas = subir_artistas('resultado_indie.csv', engine)

✅ Se han subido 379 artistas nuevos.


In [ ]:
df_maestro_artistas = subir_artistas('resultado_rock.csv', engine)

✅ Se han subido 495 artistas nuevos.


In [ ]:
df_maestro_artistas = subir_artistas('resultado_hiphop.csv', engine)

✅ Se han subido 769 artistas nuevos.


In [ ]:
# Lista de los archivos de generos
archivos_a_subir = ['lista_reggaeton.csv', 'lista_indie.csv', 'lista_latin.csv', 'lista_rock.csv', 'lista_hiphop.csv']

In [ ]:
procesar_y_subir_tracks(archivos_a_subir, engine)                      # Subir canciones

Obteniendo IDs de artistas desde SQL...
Procesando lista_reggaeton.csv...
✅ 150 canciones nuevas subidas desde lista_reggaeton.csv.
Procesando lista_indie.csv...
✅ 987 canciones nuevas subidas desde lista_indie.csv.
Procesando lista_latin.csv...
✅ 1524 canciones nuevas subidas desde lista_latin.csv.
Procesando lista_rock.csv...
✅ 1111 canciones nuevas subidas desde lista_rock.csv.
Procesando lista_hiphop.csv...
✅ 1864 canciones nuevas subidas desde lista_hiphop.csv.
